In [ ]:
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Import data
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail()

In [ ]:
dataset

In [ ]:
dataset.info()

In [ ]:
# Convert categorical columns to numeric values

# Binary encode 'sex' and 'smoker'
dataset['sex'] = dataset['sex'].map({'male': 1, 'female': 0})
dataset['smoker'] = dataset['smoker'].map({'yes': 1, 'no': 0})

# One-hot encode 'region' and drop the first category to avoid dummy variable trap
dataset = pd.get_dummies(dataset, columns=['region'], drop_first=True)

# Display the first few rows of the updated dataset
dataset.head()


In [ ]:
# Split the dataset into training (80%) and testing (20%) sets
from sklearn.model_selection import train_test_split

train_dataset, test_dataset = train_test_split(dataset, test_size=0.2, random_state=42)

# Separate the target (label) column "expenses"
train_labels = train_dataset.pop('expenses')
test_labels = test_dataset.pop('expenses')

# Display shapes to verify
print("Train dataset shape:", train_dataset.shape)
print("Test dataset shape:", test_dataset.shape)
print("Train labels shape:", train_labels.shape)
print("Test labels shape:", test_labels.shape)


In [ ]:
train_dataset

In [ ]:
train_labels

In [ ]:
# Normalize only the continuous numeric features
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# Columns to normalize
numeric_features = ['age', 'bmi', 'children']

# Fit on training data and transform both train and test sets
train_dataset[numeric_features] = scaler.fit_transform(train_dataset[numeric_features])
test_dataset[numeric_features] = scaler.transform(test_dataset[numeric_features])

# Display preview
train_dataset.head()


In [ ]:
train_labels

In [ ]:
# Build and train a simple neural network regression model

tf.random.set_seed(42)
np.random.seed(42)

# Define the model
model = keras.Sequential([
    layers.Input(shape=(train_dataset.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)  # linear output for regression
])

In [ ]:
# Compile the model
model.compile(optimizer='adam',
              loss='mse',
              metrics=['mae', 'mse'])

# Early stopping to prevent overfitting
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

In [ ]:
# Train the model
history = model.fit(
    train_dataset,
    train_labels,
    validation_split=0.2,
    epochs=500,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0  # set to 1 if you want to see progress
)

# Show a quick summary
model.summary()

In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)


# Source

This is my solution for a training project from FreeCodeCamp():
https://www.freecodecamp.org/learn/machine-learning-with-python/machine-learning-with-python-projects/linear-regression-health-costs-calculator